# 天气与特征：本轮生成、可用时点及覆盖

本页读取已归档特征审计与字典。以下方法契约解释不同信息组；是否实际入模以本轮 manifest 的特征清单为准，不把尚未启用的信息组说成已完成实验。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

if "ROOT" not in globals():
    ROOT = Path.cwd()
    while not (ROOT / "scripts/build_features.py").exists():
        if ROOT.parent == ROOT:
            raise RuntimeError("请在 Airline-delay 仓库中运行 Notebook")
        ROOT = ROOT.parent
if "RUN_DIR" not in globals():
    candidates = [p for p in (ROOT / "experiments/runs").glob("R*")
                  if (p / "metrics.csv").exists() and (p / "model_manifest.json").exists()]
    if not candidates:
        raise RuntimeError("请先归档预测实验，或手动设置 RUN_DIR")
    RUN_DIR = max(candidates, key=lambda p: int(p.name[1:]))
TABLES = RUN_DIR / "tables"
FIGURES = RUN_DIR / "figures"
def table(name):
    return pd.read_csv(TABLES / name)
def figure(name):
    display(Image(filename=str(FIGURES / name)))
def receipt(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))
display(Markdown(f"**归档运行：{RUN_DIR.name}**。本 Notebook 读取该轮表格与图，不重新拟合模型。"))

In [ ]:
config = receipt("config.json") if (RUN_DIR / "config.json").exists() else {}
model_manifest = receipt("model_manifest.json")
features = model_manifest.get("features", config.get("features", []))
group_name = model_manifest.get("feature_group", config.get("feature_group", "未登记"))
models = model_manifest.get("models", [])
display(Markdown(f"本轮信息组：**{group_name}**；阶段：**{config.get('stage', '未登记')}**；模型：**{', '.join(models)}**。"))
display(pd.DataFrame([{"development_only": model_manifest.get("development_only"),
                       "parent_run_id": config.get("parent_run_id"),
                       "change": config.get("change"), "include_baselines": model_manifest.get("include_baselines")}]))

## 1. 实际特征及生成审计

目标与标签时刻是评价元数据，不是预测变量。只在训练折拟合插补、类别处理和缩放；缺失不能按无延误、晴天或已知零填充。

In [ ]:
display(pd.DataFrame({"实际入模字段": features}))
if (RUN_DIR / "feature_audit.json").exists():
    audits = receipt("feature_audit.json")
    for section, values in audits.items():
        display(Markdown(f"**{section}**"))
        display(pd.json_normalize(values) if isinstance(values, dict) else values)
else:
    display(Markdown("本轮未归档 feature_audit.json。"))
dictionary_path = RUN_DIR / "feature_dictionary.csv"
if dictionary_path.exists():
    display(pd.read_csv(dictionary_path))
else:
    display(Markdown("本轮未归档特征字典。"))

## 2. 航班链、飞机映射及机场运行

航班链历史默认使用预测前 24 小时已完成的航段，不重复计入当前上游；其已知到达延误单独作为字段。已知尾号没有观察到有效历史时，计数为零、均值缺失；未知尾号保留缺失。history_window_incomplete 是**已知左截断指示**，不是 flag=0 时的完整覆盖保证。源数据按当地 4 月起飞日期取样，前月起飞而延迟到达/出发的 carry-in 航班可能缺失。

B43 是静态参考清单，以航司＋尾号连接，不能当作逐预测时点的历史机队快照。family 与 variant 单独比较，含截距线性模型不同时冗余展开二者。

机场历史窗口是 \( [t-60\mathrm{min},t) \)，只计已实际出发事件；计划负荷窗口是 \( [t,t+60\mathrm{min}) \)。最终取消状态不能用于事后删去计划负荷。计划快照事前已知是研究假设，仓库没有逐时计划修订版本。

## 3. 天气时点契约

METAR 假设观测后 5 分钟可用，观测年龄最多 120 分钟；这是建模假设，不是源数据提供的接收时间。TAF 先选预测时点已可用的最新公告，再匹配查询时刻 max(下游计划出发, prediction_time)，并处理有效期、修订、取消/NIL 及临时/概率段。NOAA 仅作独立补缺分支，保留质量码、单位核对和逐值来源。缺天气保留样本及缺失标记，覆盖对照须核对评估 ID。

## 4. 实际缺失、映射与覆盖表

以下仅展示本轮确实归档的表。没有对应表不代表该信息源覆盖完整，也不代表该信息组已经启用。

In [ ]:
coverage_paths = sorted({p for folder in [RUN_DIR, TABLES] if folder.exists()
                         for p in folder.glob("*.csv")
                         if any(word in p.stem.lower() for word in ["missing", "coverage", "match_rate", "unmatched"])})
for path in coverage_paths:
    display(Markdown(f"**{path.relative_to(RUN_DIR)}**"))
    data = pd.read_csv(path)
    display(Markdown(f"归档表共 {len(data):,} 行；下方展示前 50 行。"))
    display(data.head(50))
if not coverage_paths:
    display(Markdown("本轮没有单独归档缺失/覆盖表；请结合上方审计阅读。"))